# PixCell Multi-Channel ControlNet — Colab Training (Stages 0–2)

This notebook runs the computationally expensive stages of the PixCell ControlNet pipeline on Google Colab:

| Stage | Script | What it does |
|-------|--------|--------------|
| **0** | `stage0_setup.py` | Download pretrained models (PixCell-256, ControlNet, UNI-2h, SD3.5 VAE) |
| **1** | `stage1_extract_features.py` | Extract UNI embeddings + VAE latents from H&E and cell-mask images |
| **2** | `stage2_train.py` | Train ControlNet + TME module on paired ORION-CRC data |

Stage 3 (inference) is lightweight and can run locally.

## Environment Setup

Import core libraries and set up the Colab environment.

In [1]:
import os
import numpy as np
import torch
from IPython.display import clear_output
from google.colab import userdata

clear_output()

## Clone Repository & Install Dependencies

Clone the PixCell repo, check out the ControlNet branch, and install Python packages.

In [2]:
%cd /content
!git clone https://github.com/pohaoc2/PixCell.git
%cd /content/PixCell
!git checkout main
!pip install -r requirements.txt
!pip install mmcv==1.7.0
clear_output()

## Configure AWS & HuggingFace Credentials

Set AWS credentials (for S3 data access) and HuggingFace token (for gated model downloads).
These should be stored in Colab's **Secrets** tab (`userdata`).

In [3]:
!pip install awscli
clear_output()

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Download ORION-CRC Data from S3

Download the paired experimental dataset (H&E tiles + multi-channel TME maps) into `data/orion-crc/`.
The dataset contains:
- `he/` — H&E image tiles (256×256 PNG)
- `exp_channels/` — per-channel TME maps (cell_mask, cell_type_*, cell_state_*, vasculature, oxygen, glucose)

> **Note:** Update `S3_DATA_PATH` with the actual S3 URI before running.

In [4]:
%cd /content/PixCell

S3_DATA_PATH = "s3://bagherilab-working/pohao/share_space/orion-crc33.zip"

!aws s3 cp {S3_DATA_PATH} ./data/orion-crc33.zip
!unzip ./data/orion-crc33.zip -d ./data/orion-crc33

clear_output()

In [6]:
!mv data/orion-crc33/ data/orion-crc33-test
!mv data/orion-crc33-test/orion-crc33 data/

In [7]:
print("H&E tiles:", len(os.listdir("data/orion-crc33/he")))
print("Channel dirs:", os.listdir("data/orion-crc33/exp_channels"))

H&E tiles: 10379
Channel dirs: ['cell_state_nonprolif', 'glucose', 'cell_types', 'cell_type_immune', 'cell_states', 'cell_state_dead', 'oxygen', 'cell_type_healthy', 'cell_type_cancer', 'cell_state_prolif', 'cell_masks', 'vasculature']


## Stage 0 — Download Pretrained Models

Download all four pretrained model components into `pretrained_models/`:
- **PixCell-256** transformer (frozen base model)
- **PixCell-256 ControlNet** (initialization weights)
- **UNI-2h** (histopathology feature extractor)
- **SD 3.5 VAE** (image encoder/decoder)

Requires `HF_TOKEN` for gated model access.

In [5]:
%cd /content/PixCell
!python stage0_setup.py --token {os.environ['HF_TOKEN']}
clear_output()
print("Pretrained models:")
!ls pretrained_models/

Pretrained models:
pixcell-256  pixcell-256-controlnet  sd-3.5-vae  uni-2h


## Stage 1 — Extract Features (Pass 1: H&E Images)

Extract **UNI-2h embeddings** (`*_uni.npy`) and **SD3.5 VAE latents** (`*_sd3_vae.npy`) from H&E tiles.
These are cached once and loaded at every training step.

In [ ]:
%cd /content/PixCell
!python stage1_extract_features.py \
    --image-dir  ./data/orion-crc33/he \
    --output-dir ./data/orion-crc33/features

## Stage 1 — Extract Features (Pass 2: Cell Masks)

Extract **VAE latents for cell_mask images** (`*_mask_sd3_vae.npy`).
These are used as additional ControlNet conditioning during training. UNI extraction is skipped for masks.

In [ ]:
%cd /content/PixCell
!python stage1_extract_features.py \
    --image-dir   ./data/orion-crc33/exp_channels/cell_mask \
    --output-dir  ./data/orion-crc33/vae_features \
    --vae-prefix  mask_sd3_vae \
    --skip-uni

## Stage 1 — Build HDF5 Metadata Index

Create the HDF5 index file that maps tile IDs to their channel/feature paths.
This index is loaded by the dataloader during training.

In [8]:
from diffusion.data.datasets.paired_exp_controlnet_dataset import build_exp_index

build_exp_index(
    exp_channels_dir="data/orion-crc33/exp_channels",
    output_path="data/orion-crc33/metadata/exp_index.hdf5",
)

[build_exp_index] Wrote 10379 IDs → data/orion-crc33/metadata/exp_index.hdf5


## Stage 2 — Train ControlNet

Train the ControlNet + TME conditioning module on paired ORION-CRC data.
Config is in `configs/config_controlnet_exp.py` (edit `exp_data_root`, `num_epochs`, etc. as needed).

Key training features:
- CFG dropout (15%) enables TME-only inference
- Channel reliability weights attenuate approximate channels (vasculature, oxygen, glucose)

In [ ]:
%cd /content/PixCell
!python stage2_train.py configs/config_controlnet_exp.py

/content/PixCell
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
2026-03-21 05:22:23,693 - PixArt - INFO - Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: no

2026-03-21 05:22:23,723 - PixArt - INFO - Config: 
data_root = './'
data = dict

## Upload Checkpoint to S3

Save the trained ControlNet checkpoint to S3 for later use in Stage 3 (inference).
Update the checkpoint path below to match the latest training step.

In [ ]:
ckpt_dir = "/content/PixCell/checkpoints/pixcell_controlnet_exp/checkpoints"
print("Available checkpoints:")
!ls {ckpt_dir}

ckpt_path = f"{ckpt_dir}/step_XXXXXXX"  # TODO: replace with actual step directory
!aws s3 cp {ckpt_path} s3://bagherilab-working/pohao/share_space/checkpoints/ --recursive

## Disconnect Runtime

Release the Colab GPU once training and upload are complete.

In [ ]:
from google.colab import runtime
runtime.unassign()